# Workshop 7. Multiple regression and checking the model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aaubs/AQM-2026/blob/main/lecture07/workshop07.ipynb)

In this workshop you write the code yourselves. Work in your group or alone, and ask the TAs when you are stuck.

The tasks follow the routine from the lecture: fit, check, fix, extend, check again. Task 1 is the lecture's
multiple regression in code, fully worked: run it, answer the questions and change one thing. Tasks 2 to 4 start
from an empty cell: checking a model and responding to what you find, visits in levels or in logs, and predictions
for two customers. The extension tasks are for those who have time left. Exercises in the style of the textbook, with
small numbers and no data, are in the TA's separate notebook for this session (link on Moodle).

Each core task has two hints: the first names the statistical step, the second names the functions. Try the task
before you open them. The toolbox says where you have seen each function before, so you can look it up in that
notebook. The code patterns, the slides and the lecture notebook are your main references. You are welcome to use an
LLM, but every task can be solved without one, and you should be able to explain the statistics in your answers
yourself.

In every task, say what y and the x's are, and in which units, before you fit anything. Each answer is in words a
manager can follow, with the numbers it rests on.

## Toolbox

The functions you need today, and where you have seen them in use.

| Function | What it does | Seen in |
|---|---|---|
| `smf.ols("y ~ x1 + x2", data=df).fit()` | Fits a regression with several x's | Lecture 7, Part 3 |
| `.fit(cov_type="HC1")` | The same model with robust standard errors | Lecture 7, Part 2 |
| `model.params`, `model.bse`, `model.conf_int()` | Coefficients, standard errors, 95% intervals | Lecture 6, Parts 1 and 3 |
| `model.rsquared_adj` | Adjusted R² | Lecture 7, Part 3 |
| `model.fittedvalues`, `model.resid` | ŷ and the residuals e, one per row | Lecture 7, Part 1 |
| `het_breuschpagan(model.resid, model.model.exog)` | Breusch-Pagan test: LM, its p-value, an F version, its p-value | Lecture 7, Part 1 |
| `stats.skew(r)`, `stats.kurtosis(r, fisher=False)` | Skewness (0 if normal) and kurtosis (3 if normal) | Lecture 7, Part 1 |
| `variance_inflation_factor(X, i)` with `X = model.model.exog` | VIF of the x in column i (column 0 is the intercept) | Lecture 7, Part 3 |
| `np.log(...)`, `np.exp(...)` | Natural log, and back | Lecture 7, Parts 2 and 4 |
| `model.predict(pd.DataFrame({...}))` | Predictions; one column for every x in the formula | Lecture 6, Part 4; Lecture 7, Part 4 |
| `df.boxplot(column="col", by="group")` | Box plots of a column for each group | Lecture 5 |
| `plt.scatter(x, y)`, `plt.axhline(0)` | Scatter plot; a horizontal line at 0 | Lecture 7, Part 1 |

## Code patterns

Two complete patterns for today. Replace `df`, `y`, `x1` and `x2` with your data and column names. The patterns are
templates; they do not run as they stand.

**Fit, check, and refit with robust standard errors**

```python
model = smf.ols("y ~ x1 + x2", data=df).fit()

plt.scatter(model.fittedvalues, model.resid, s=8, alpha=0.5)    # residual plot
plt.axhline(0, color="darkorange", linestyle="--")
plt.xlabel("Fitted value"); plt.ylabel("Residual")
plt.show()

lm, lm_p, f, f_p = het_breuschpagan(model.resid, model.model.exog)
print(f"Breusch-Pagan = {lm:.2f}, p = {lm_p:.2g}")
print(f"Skew = {stats.skew(model.resid):.2f}, Kurtosis = {stats.kurtosis(model.resid, fisher=False):.2f}")

robust = smf.ols("y ~ x1 + x2", data=df).fit(cov_type="HC1")
print(pd.DataFrame({"coef": model.params, "se classical": model.bse, "se robust": robust.bse}))
```

**A log model, a prediction, and back to the original unit**

```python
df["log_y"] = np.log(df["y"])
model = smf.ols("log_y ~ x1 + x2", data=df).fit(cov_type="HC1")
print(model.summary())
b = model.params["x1"]
print(f"about {100 * b:.1f}%, exactly {100 * (np.exp(b) - 1):.1f}% per unit of x1")

new = pd.DataFrame({"x1": [5], "x2": [40]})      # one column for every x in the formula
print(np.exp(model.predict(new)))                # predicted y in the original unit
```

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor

url = "https://raw.githubusercontent.com/aaubs/AQM-2026/main/data/"
customers = pd.read_csv(url + "customers_sample.csv")   # 1,000 RetailCorp customers drawn at random
customers["log_spend"] = np.log(customers["spend_2025"])
customers["log_visits"] = np.log(customers["visits_2025"])
customers.head()

## Core tasks

### Task 1. Tenure among customers of the same age (worked)

The lecture's multiple regression in code. Run the cells and answer the questions below them.

y = `log_spend` (ln of spend in DKK in 2025); x<sub>1</sub> = `tenure_years` (years as a customer); x<sub>2</sub> =
`age` (years).

In [ ]:
y_col = "log_spend"      # step 3: change to "spend_2025"

simple = smf.ols(f"{y_col} ~ tenure_years", data=customers).fit()
multiple = smf.ols(f"{y_col} ~ tenure_years + age", data=customers).fit()
print(multiple.summary())

In [ ]:
b_simple = simple.params["tenure_years"]
b1, b2 = multiple.params["tenure_years"], multiple.params["age"]
d = smf.ols("age ~ tenure_years", data=customers).fit().params["tenure_years"]

print(f"Tenure alone: {b_simple:.4f}   with age: {b1:.4f}   age: {b2:.4f}")
print(f"b1 + b2 * d = {b1:.4f} + {b2:.4f} * {d:.4f} = {b1 + b2 * d:.4f}")
print(f"The tenure coefficient falls to {b1 / b_simple:.0%} of its simple value")
if y_col == "log_spend":
    print(f"Tenure with age, as a percent: about {100 * b1:.1f}%, exactly {100 * (np.exp(b1) - 1):.1f}% per year")

**Questions**

1. Read the tenure coefficient with age in the model as a percent, and give its 95% interval from the output. What
   does "among customers of the same age" mean here?
2. Why does the tenure coefficient fall when age is added? Use the line b̃<sub>1</sub> = b<sub>1</sub> +
   b<sub>2</sub> d that the second cell prints.
3. Is age itself clearly different from 0? Where in the output do you see it?

**Change one thing.** In the first cell, change `y_col = "log_spend"` to `"spend_2025"` and run both cells. The
coefficients are now in DKK. Does age change the tenure coefficient by about the same share in levels as in logs?

**Your answer:**

### Task 2. Check the levels model, and respond

Fit spend in DKK (`spend_2025`) on `tenure_years` and `age`, and check the model as in Part 1 of the lecture.

1. Make the residual plot. Is there a curve? A fan?
2. Run the Breusch-Pagan test, and compute the skewness and kurtosis of the residuals. Which assumptions hold, and
   which fail?
3. Make box plots of the residuals by `region`. Does the scatter around the line differ between the regions?
4. Refit the model with robust standard errors (HC1). What changes, and what does not? Does your conclusion about
   tenure change?
5. Fit the same model with ln spend as y and run the Breusch-Pagan test again. What do you find?

<details>
<summary>Hint 1</summary>

The checks use the residuals of the levels model. For question 3, put the residuals in a column of the data first.
Robust standard errors leave the coefficients unchanged; compare the standard errors, t and the intervals.
</details>

<details>
<summary>Hint 2</summary>

The first code pattern does questions 1, 2 and 4. For question 3: `customers["resid"] = model.resid`, then
`customers.boxplot(column="resid", by="region")`. For question 5: `smf.ols("log_spend ~ tenure_years + age", ...)`
and `het_breuschpagan` again.
</details>

**Your answer:**

### Task 3. Visits in levels or in logs

Should visits enter the model in levels or in logs? Fit two models for ln spend, both with `tenure_years` and `age`:
one with `visits_2025` and one with `log_visits`.

1. Compare adjusted R² of the two models. Why is the comparison allowed here, when R² of a levels and a log model for
   spend could not be compared in the lecture?
2. Read the coefficient on `log_visits` as an elasticity, with its 95% interval.
3. In Exercise 2 of the lecture, ln spend on ln visits alone gave an elasticity of 0.294. Why is it larger with
   tenure and age in the model?

<details>
<summary>Hint 1</summary>

Adjusted R² compares models for the same y and the same customers. An elasticity is the percent difference in y per
1 percent difference in x. For question 3, think of what was left out in Exercise 2 and how it relates to visits and
to spend.
</details>

<details>
<summary>Hint 2</summary>

`smf.ols("log_spend ~ tenure_years + age + visits_2025", ...)` and `... + log_visits`, `.rsquared_adj`,
`.params`, `.conf_int()`, and `customers[["age", "log_visits", "log_spend"]].corr()`.
</details>

**Your answer:**

### Task 4. Predictions for two customers

Use the final model from the lecture: ln spend on `tenure_years`, `age` and `visits_2025`. Predict the spend in DKK of
two customers:

| Customer | Years as a customer | Age | Visits in 2025 |
|---|---|---|---|
| A | 1 | 25 | 30 |
| B | 8 | 60 | 20 |

1. Give the predicted ln spend and the predicted spend in DKK for each.
2. How many times as much does the model predict for B as for A? Show that you get the same number from the
   difference in predicted ln spend.
3. Compare the predictions with the mean and the median spend in the sample. Why is the exponential of a predicted ln
   spend a typical spend for such customers, and not their average spend?
4. Are A and B inside the data? Count the customers with at most 2 years and at most 30 years of age, and those with
   at least 6 years and at least 55 years of age.

<details>
<summary>Hint 1</summary>

The model predicts ln spend; the exponential takes it back to DKK. A difference in ln is a ratio in the original
unit: exp(ln y<sub>B</sub> − ln y<sub>A</sub>) = y<sub>B</sub> / y<sub>A</sub>.
</details>

<details>
<summary>Hint 2</summary>

`pd.DataFrame({"tenure_years": [1, 8], "age": [25, 60], "visits_2025": [30, 20]})`, `model.predict(...)`,
`np.exp(...)`, `customers["spend_2025"].mean()` and `.median()`, and `((cond1) & (cond2)).sum()` to count rows.
</details>

**Your answer:**

## Extension tasks

### Extension 1. Age when they became customers

Make a new column with each customer's age at the first order, `age_first = age - tenure_years`, and add it to the
model for ln spend with `tenure_years` and `age`. Look at the coefficients and at the notes under the output. What
happened, and why can the data not separate the three variables?

**Your answer:**

### Extension 2. The VIF by hand

For the model of ln spend on `tenure_years` and `age`, regress `age` on `tenure_years`, take its R², and compute
VIF = 1 / (1 − R²). Check the result with `variance_inflation_factor`. Then do the same for the model that also has
`visits_2025`: regress each x on the other two.

**Your answer:**